# Stage 3 — Symptom Tree

Builds a symptom tree from the **latest admission note** + IE + prior admission history. Uses the
diagnosis-redacted note from Stage 1.5 (not the raw cohort note), same as Stage 2.

**Next:** `stage_04_export_patient_records.ipynb`


## Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / "pipeline.py").exists():
    NB_DIR = ROOT
elif (ROOT / "notebooks" / "pipeline.py").exists():
    NB_DIR = ROOT / "notebooks"
else:
    NB_DIR = ROOT.parent / "notebooks"
sys.path.insert(0, str(NB_DIR))

import pandas as pd
from pipeline import (
    LLMNotAvailableError,
    check_llm,
    get_llm_config,
    load_cohort,
    load_ie_results,
    load_redaction_results,
    print_pipeline_banner,
    save_symptom_tree_results,
    symptom_tree_agent,
)

print_pipeline_banner()
LLM_CONFIG = get_llm_config(for_symptom_tree=True)
ok, model_info = check_llm(LLM_CONFIG)
if not ok:
    raise LLMNotAvailableError(model_info)
print(f"LLM ready — {LLM_CONFIG.method_prefix()}: {model_info}")

cohort_df = load_cohort()

# Stage 1.5 strips diagnosis information from the latest admission's note — the symptom tree
# must be built from that redacted note, not the raw cohort note. Run stage_01b first.
redaction_df = load_redaction_results()
redacted_lookup = redaction_df.set_index("patient_id")["redacted_note"].to_dict()
cohort_df["clinical_note_original"] = cohort_df["clinical_note"]
cohort_df["clinical_note"] = cohort_df["patient_id"].astype(str).map(redacted_lookup)
missing = cohort_df["clinical_note"].isna()
if missing.any():
    raise ValueError(
        f"{missing.sum()} patient(s) missing redacted notes — run "
        "notebooks/stage_01b_redact_notes.ipynb for the full cohort first."
    )

ie_df = load_ie_results()
merged = cohort_df.merge(ie_df, on="hadm_id", how="inner", suffixes=("", "_ie"))
print(f"Patients to process: {len(merged)} — using diagnosis-redacted clinical notes (Stage 1.5)")


## Build Patient Symptom Trees (latest admission)

In [ ]:
trees = []

for _, row in merged.iterrows():
    hadm_id = str(row["hadm_id"])
    patient_id = str(row["patient_id"])
    history = row.get("admission_history") or row.get("admission_history_ie") or []
    extracted = row["extracted"]
    print(f"Symptom tree patient={patient_id} hadm_id={hadm_id} ({len(history)} prior)...")
    ctx = row.get("clinical_context_text") or row.get("clinical_context_text_ie") or ""
    tree = symptom_tree_agent(
        clinical_note=row["clinical_note"],
        extracted=extracted,
        admission_id=hadm_id,
        patient_id=patient_id,
        config=LLM_CONFIG,
        admission_history=history,
        clinical_context_text=ctx,
    )
    trees.append({
        "patient_id": patient_id,
        "admission_id": row["admission_id"],
        "subject_id": int(row["subject_id"]),
        "hadm_id": int(row["hadm_id"]),
        "n_prior_admissions": len(history),
        "admission_history": history,
        "primary_icd_code": row["primary_icd_code"],
        "primary_dx_title": row["primary_dx_title"],
        "ground_truth_icd10": row["ground_truth_icd10"],
        "ground_truth_dx_titles": row["ground_truth_dx_titles"],
        "n_diagnoses": int(row["n_diagnoses"]),
        "extraction_method": row["extraction_method"],
        "extracted": extracted,
        "symptom_tree_method": tree.get("_method", "unknown"),
        "symptom_tree": tree,
        "branch_count": len(tree.get("branches", [])),
    })

results_df = pd.DataFrame(trees)
results_df[["patient_id", "admission_id", "n_prior_admissions", "symptom_tree_method", "branch_count"]]



## Save Results

In [ ]:
patient_symptom_trees = {
    str(r["patient_id"]): r["symptom_tree"] for _, r in results_df.iterrows()
}

out = save_symptom_tree_results(results_df, patient_symptom_trees)
print(f"Saved → {out}")
